<a href="https://www.kaggle.com/code/ameythakur20/llm-classification-inference" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# LLM Classification Finetuning: Ensembled Pipeline Inference

**Kaggle Competition Solution**

**Author:** [Amey Thakur](https://www.kaggle.com/ameythakur20)

This architecture predicts human preference in responses from large language models.
The procedure involves ensembling predictions from multiple base models (Gemma-2-9B and
Llama-3-8B). Since these are high-capacity models requiring substantial memory, we utilize
a pipeline parallel inference schema that splits the layers across the two available 15GB
T4 GPUs. This ensures stable runtime under operational limits without triggering Out Of
Memory (OOM) errors.

The solution applies test time augmentation implicitly by running the Llama model on
swapped versions of the response datasets and re ordering the resulting probabilities.

**Outline:**

1. [System Dependencies and Offline Modules](#1-system-dependencies-and-offline-modules)
2. [Data Structuring and Preprocessing](#2-data-structuring-and-preprocessing)
3. [Gemma-2 Inference Pipeline](#3-gemma-2-inference-pipeline)
4. [Llama-3 Swapped Inference Pipeline](#4-llama-3-swapped-inference-pipeline)
5. [Logit Interpolation and Final Submission](#5-logit-interpolation-and-final-submission)
6. [Summary](#6-summary)

---

## 1. System Dependencies and Offline Modules

> **[CRITICAL RESOURCE INSTRUCTION]**
> Kaggle's Inference requires internet to be **Disabled**. You must attach the following external Kaggle datasets to your Notebook environment for this code to run:
> 1. `lmsys-packages` (Contains offline `.whl` files for Triton & Xformers)
> 2. `lmsys-modules-0805` (Contains the `human_pref` codebase module)
> 3. `lmsys-checkpoints-0-0805` (Contains Gemma-2-9B model weights)
> 4. `lmsys-checkpoints-3-0805` (Contains Llama-3-8B model weights)
> 
> *You can search for these dataset names in the "Add Input > Public Datasets" menu on Kaggle and attach them.*

Hardware limitations necessitate using optimized attention kernels. We install
Triton and Xformers from offline wheels. The local `human_pref` module contains
specialized collators and model definitions adapted for pipeline parallel execution.

In [ ]:
import os
import shutil
import warnings

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

mod_paths = [
    "/kaggle/input/datasets/tascj0/lmsys-modules-0805",
    "/kaggle/input/lmsys-modules-0805"
]
source_dir = next((p for p in mod_paths if os.path.exists(p)), None)

if source_dir:
    if os.path.exists("human_pref"):
        shutil.rmtree("human_pref")
    shutil.copytree(source_dir, "human_pref")

    # Global Native Patches to bypass Python 3.10/3.12 Triton conflicts
    ops_dir = "human_pref/models/ops"
    
    # 1. Patch rms_norm.py
    with open(f"{ops_dir}/rms_norm.py", "w") as f:
        f.write("import torch\ndef rms_norm(x, weight, eps=1e-6):\n    x_f = x.float()\n    v = x_f.pow(2).mean(-1, keepdim=True)\n    return (x_f * torch.rsqrt(v+eps) * weight.float()).to(x.dtype)\n")

    # 2. Patch gelu_and_mul.py
    with open(f"{ops_dir}/gelu_and_mul.py", "w") as f:
        f.write("import torch\ndef gelu_and_mul_fwd(x):\n    d = x.shape[-1]//2\n    return torch.nn.functional.gelu(x[...,:d]) * x[...,d:]\n")

    # 3. Patch silu_and_mul.py
    with open(f"{ops_dir}/silu_and_mul.py", "w") as f:
        f.write("import torch\ndef silu_and_mul_fwd(x):\n    d = x.shape[-1]//2\n    return torch.nn.functional.silu(x[...,:d]) * x[...,d:]\n")

    print("Inference environment stabilized with native operations.")
else:
    raise FileNotFoundError("Could not find required modules in /kaggle/input.")


## 2. Data Structuring and Preprocessing

We prepare two data structures for inference:
1. A standard representation for the first model.
2. A transposed representation where Response A and Response B are interchanged.
This provides a variant view for the Llama model, increasing ensemble robustness
to positional bias. We perform generation as a separate python script process to
enforce absolute memory reclamation upon exit.

In [ ]:
%%writefile prepare_datasets.py
import pandas as pd

# Read the testing sequence inputs
df = pd.read_csv("/kaggle/input/competitions/llm-classification-finetuning/test.csv")

# Initialize target columns which the SequenceClassification processor requires
df["winner_model_a"] = 1
df["winner_model_b"] = 0
df["winner_tie"]     = 0
df.to_parquet("test.parquet", index=False)

# Transpose responses to create an inverse view for the second model
df["response_a"], df["response_b"] = df["response_b"], df["response_a"]
df.to_parquet("test_swap.parquet", index=False)
print("Dataset representations generated successfully.")

In [ ]:
!python prepare_datasets.py

## 3. Gemma-2 Inference Pipeline

The Gemma-2-9B model logic is delegated to a standalone script.
We employ manual layer allocation (device_map) across cuda:0 and cuda:1.
The dataloader uses a specialized `VarlenCollator` combined with max token
sharding to prevent extreme memory spikes during batching. Utilizing a subprocess
circumvents IPython notebook memory retention issues.

In [ ]:
%%writefile predict_gemma.py
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer

from xformers.ops.fmha.attn_bias import BlockDiagonalCausalMask
from human_pref.models.modeling_gemma2 import Gemma2ForSequenceClassification
from human_pref.data.processors import ProcessorPAB
from human_pref.data.dataset import LMSYSDataset
from human_pref.data.collators import VarlenCollator, ShardedMaxTokensCollator
import os
from human_pref.utils import to_device

possible_checkpoints = [
    "/kaggle/input/datasets/tascj0/lmsys-checkpoints-0-0805",
    "/kaggle/input/lmsys-checkpoints-0-0805"
]
model_name_or_path = next((p for p in possible_checkpoints if os.path.exists(p)), possible_checkpoints[0])
csv_path = "test.parquet"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
processor = ProcessorPAB(
    tokenizer=tokenizer,
    max_length=4096,
    support_system_role=False,
)
dataset = LMSYSDataset(
    csv_file=csv_path,
    query=None,
    processor=processor,
    include_swap=False,
    is_parquet=True,
)
dataloader = DataLoader(
    dataset,
    batch_size=80,
    num_workers=4,
    collate_fn=ShardedMaxTokensCollator(
        max_tokens=8192, base_collator=VarlenCollator()
    ),
)

# Allocate hidden layers evenly across hardware to prevent OOM
num_hidden_layers = 42
device_map = {
    "model.embed_tokens": "cuda:0",
    "model.norm": "cuda:1",
    "score": "cuda:1",
}
for i in range(num_hidden_layers // 2):
    device_map[f"model.layers.{i}"] = "cuda:0"
for i in range(num_hidden_layers // 2, num_hidden_layers):
    device_map[f"model.layers.{i}"] = "cuda:1"

model = Gemma2ForSequenceClassification.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float16,
    device_map=device_map,
)

# Generate positional parameters per device
config = model.config
dim = config.head_dim
inv_freq = 1.0 / (
    config.rope_theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim)
)
inv_freq0 = inv_freq.to("cuda:0")
inv_freq1 = inv_freq.to("cuda:1")

is_first = True
hidden_states = None
outs = []

# Execute dual device synchronization loops
for batch in tqdm(dataloader, desc="Gemma Inference"):
    for micro_batch in batch:
        input_ids = to_device(micro_batch["input_ids"], "cuda:0")
        seq_info = {
            "cu_seqlens": micro_batch["cu_seqlens"],
            "position_ids": micro_batch["position_ids"],
            "max_seq_len": micro_batch["max_seq_len"],
            "attn_bias": BlockDiagonalCausalMask.from_seqlens(micro_batch["seq_lens"]),
        }
        seq_info = to_device(seq_info, "cuda:0")

        if is_first:
            with torch.no_grad(), torch.cuda.amp.autocast():
                prev_hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)
            is_first = False
            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, prev_hidden_states], "cuda:1"
            )
            continue

        with torch.no_grad(), torch.cuda.amp.autocast():
            logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
            hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)

            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, hidden_states], "cuda:1"
            )
            outs.append(logits.cpu())

# Resolve terminal micro batch computation
with torch.no_grad(), torch.cuda.amp.autocast():
    logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
    outs.append(logits.cpu())

# Consolidate spatial predictions
pred = torch.cat(outs, dim=0)
prob = pred.softmax(-1)

np.save("prob_gemma.npy", prob.numpy())
print("Gemma pipeline successfully finalized.")

In [ ]:
!python predict_gemma.py

## 4. Llama-3 Swapped Inference Pipeline

The structural foundation relies identically on pipeline partitioning, optimized 
here for Llama architecture dimension scaling. We operate this pass on the 
`test_swap.parquet` dataset to counteract token occurrence biases.

In [ ]:
%%writefile predict_llama.py
import torch
import numpy as np
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer

from xformers.ops.fmha.attn_bias import BlockDiagonalCausalMask
from human_pref.models.modeling_llama import LlamaForSequenceClassification
from human_pref.data.processors import ProcessorPAB
from human_pref.data.dataset import LMSYSDataset
from human_pref.data.collators import VarlenCollator, ShardedMaxTokensCollator
import os
from human_pref.utils import to_device

possible_checkpoints = [
    "/kaggle/input/datasets/tascj0/lmsys-checkpoints-3-0805",
    "/kaggle/input/lmsys-checkpoints-3-0805"
]
model_name_or_path = next((p for p in possible_checkpoints if os.path.exists(p)), possible_checkpoints[0])
csv_path = "test_swap.parquet"

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
tokenizer.deprecation_warnings["sequence-length-is-longer-than-the-specified-maximum"] = True

processor = ProcessorPAB(
    tokenizer=tokenizer,
    max_length=4096,
    support_system_role=True,
)
dataset = LMSYSDataset(
    csv_file=csv_path,
    query=None,
    processor=processor,
    include_swap=False,
    is_parquet=True,
)
dataloader = DataLoader(
    dataset,
    batch_size=80,
    num_workers=4,
    collate_fn=ShardedMaxTokensCollator(
        max_tokens=8192, base_collator=VarlenCollator()
    ),
)

num_hidden_layers = 32
device_map = {
    "model.embed_tokens": "cuda:0",
    "model.norm": "cuda:1",
    "score": "cuda:1",
}
for i in range(num_hidden_layers // 2):
    device_map[f"model.layers.{i}"] = "cuda:0"
for i in range(num_hidden_layers // 2, num_hidden_layers):
    device_map[f"model.layers.{i}"] = "cuda:1"

model = LlamaForSequenceClassification.from_pretrained(
    model_name_or_path,
    torch_dtype=torch.float16,
    device_map=device_map,
)

config = model.config
dim = config.hidden_size // config.num_attention_heads
inv_freq = 1.0 / (
    config.rope_theta ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim)
)
inv_freq0 = inv_freq.to("cuda:0")
inv_freq1 = inv_freq.to("cuda:1")

is_first = True
hidden_states = None
outs = []

for batch in tqdm(dataloader, desc="Llama Inference"):
    for micro_batch in batch:
        input_ids = to_device(micro_batch["input_ids"], "cuda:0")
        seq_info = {
            "cu_seqlens": micro_batch["cu_seqlens"],
            "position_ids": micro_batch["position_ids"],
            "max_seq_len": micro_batch["max_seq_len"],
            "attn_bias": BlockDiagonalCausalMask.from_seqlens(micro_batch["seq_lens"]),
        }
        seq_info = to_device(seq_info, "cuda:0")

        if is_first:
            with torch.no_grad(), torch.cuda.amp.autocast():
                prev_hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)
            is_first = False
            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, prev_hidden_states], "cuda:1"
            )
            continue

        with torch.no_grad(), torch.cuda.amp.autocast():
            logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
            hidden_states = model.forward_part1(input_ids, seq_info, inv_freq0)

            prev_seq_info, prev_hidden_states = to_device(
                [seq_info, hidden_states], "cuda:1"
            )
            outs.append(logits.cpu())

with torch.no_grad(), torch.cuda.amp.autocast():
    logits = model.forward_part2(prev_hidden_states, prev_seq_info, inv_freq1)
    outs.append(logits.cpu())

pred = torch.cat(outs, dim=0)
prob = pred.softmax(-1)

np.save("prob_llama.npy", prob.numpy())
print("Llama pipeline successfully finalized.")

In [ ]:
!python predict_llama.py

## 5. Logit Interpolation and Final Submission

We amalgamate the resulting probabilistic outputs. Because the Llama model was 
subjected to structurally swapped inputs (A to B, B to A), we realign its predictions 
prior to averaging. Historical tuning confirms a weighting ratio of 0.57 for Gemma 
and 0.43 for Llama optimizes logarithmic loss evaluation boundaries.

In [ ]:
import numpy as np
import pandas as pd
import os

# Verify artifacts exist prior to merging logic
if not os.path.exists("prob_gemma.npy") or not os.path.exists("prob_llama.npy"):
    raise FileNotFoundError("Inference probability tensors are missing.")

df = pd.read_parquet("test.parquet")

prob_gemma = np.load("prob_gemma.npy")

# Geometrically map output probabilities back corresponding to the parameter swap
# Index translation: 0 (A) -> 1 (B), 1 (B) -> 0 (A), 2 (Tie) -> 2 (Tie)
prob_llama = np.load("prob_llama.npy")[:, [1, 0, 2]]

# Execute ensemble
blended_preds = np.average(
    [prob_gemma, prob_llama],
    axis=0,
    weights=[0.57, 0.43]
)

# Structure final delivery schema
submission = pd.DataFrame({
    "id": df["id"],
    "winner_model_a": blended_preds[:, 0],
    "winner_model_b": blended_preds[:, 1],
    "winner_tie": blended_preds[:, 2],
})

submission.to_csv("submission.csv", index=False)
print("Ensemble successful. Submission initialized.")
display(submission.head())

## 6. Summary

This notebook implemented an ensembled pipeline parallelism design for human preference classification:

1. **Dual Representation Generation:** We constructed standard and swapped representations of the user-response dataset, isolating test-time augmentation logic for positional bias correction.
2. **Gemma-2-9B Pipeline:** Conducted target inference against the primary representation via 42 sequentially distributed neural layers on distinct GPU partitions (cuda:0 and cuda:1).
3. **Llama-3-8B Pipeline:** Conducted auxiliary robust inference on the inverted sequence representation using 32 parallel distributed layers.
4. **Probabilistic Convergence:** Reoriented augmented probability fields prior to enforcing logarithmic convergence optimization weighting (57% Gemma-2, 43% Llama-3).

---
**Citation:**
Wei-lin Chiang, Lianmin Zheng, Lisa Dunlap, Joseph E. Gonzalez, Ion Stoica, Paul Mooney, Sohier Dane, Addison Howard, and Nate Keating. LLM Classification Finetuning. https://kaggle.com/competitions/llm-classification-finetuning, 2024. Kaggle.